# GTEx k-means parameter sweep: CLAMPbase, CLAMPfull, PLIER

Tests different combinations of:
- **Label column**: `SMTS` vs `SMTSD`
- **MIN_SAMPLES**: 50, 100, 150, 200
- **N_REPS_PER_K**: 50, 100, 200

Outputs a summary DataFrame with the best ARI for each combination.

## Libraries

In [39]:
import csv
import gc
import pickle
from itertools import product
from pathlib import Path

import cupy as cp
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from cuml.cluster import KMeans as cuKMeans
from pyprojroot.here import here
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from sklearn.metrics import adjusted_rand_score, adjusted_rand_score as ari
from sklearn.preprocessing import StandardScaler

readRDS = ro.r["readRDS"]

## Parameter grid

In [40]:
LABEL_COLS       = ["SMTS", "SMTSD"]
MIN_SAMPLES_LIST = [50, 100, 150, 200]
N_REPS_LIST      = [50, 100, 200]

# Fixed k-means settings
BASE_SEED = 10000

## Load data

In [41]:
# GTEx metadata
gtex_meta = pd.read_csv(
    here('data/gtex/GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt'),
    sep='\t',
    dtype=str,
    quoting=csv.QUOTE_NONE,
    engine='python',
    keep_default_na=False,
    on_bad_lines='warn'
).set_index('SAMPID')

print(gtex_meta.shape)
gtex_meta.head()

(22951, 62)


,SMATSSCR,SMCENTER,SMPTHNTS,SMRIN,SMTS,SMTSD,SMUBRID,SMTSISCH,SMTSPAX,SMNABTCH,...,SME1ANTI,SMSPLTRD,SMBSMMRT,SME1SNSE,SME1PCTS,SMRRNART,SME1MPRT,SMNUM5CD,SMDPMPRT,SME2PCTS
SAMPID,,,,,,,,,,,,,,,,,,,,,
GTEX-1117F-0003-SM-58Q7G,,B1,,,Blood,Whole Blood,0013756,1188,,BP-38516,...,,,,,,,,,,
GTEX-1117F-0003-SM-5DWSB,,B1,,,Blood,Whole Blood,0013756,1188,,BP-38516,...,,,,,,,,,,
GTEX-1117F-0003-SM-6WBT7,,B1,,,Blood,Whole Blood,0013756,1188,,BP-38516,...,,,,,,,,,,
GTEX-1117F-0011-R10a-SM-AHZ7F,,"B1, A1",,,Brain,Brain - Frontal Cortex (BA9),0009834,1193,,,...,,,,,,,,,,
GTEX-1117F-0011-R10b-SM-CYKQ8,,"B1, A1",,7.2,Brain,Brain - Frontal Cortex (BA9),0009834,1193,,BP-42319,...,,,,,,,,,,


In [42]:
# Helper: load B matrix from RDS
def extract_B_matrix(rds_obj):
    B_matrix = rds_obj.rx2("B")
    with localconverter(ro.default_converter + pandas2ri.converter):
        B_values = ro.conversion.rpy2py(B_matrix)
    return pd.DataFrame(
        data=B_values,
        index=B_matrix.rownames if B_matrix.rownames else None,
        columns=B_matrix.colnames if B_matrix.colnames else None,
    )


def load_rds_B_matrix(rel_path):
    return extract_B_matrix(readRDS(str(here(rel_path))))

In [43]:
# Load embeddings once
embeddings = {
    "CLAMPbase": load_rds_B_matrix('output/gtex/CLAMPbase.rds'),
    "CLAMPfull":  load_rds_B_matrix('output/gtex/CLAMPfull.rds'),
    "PLIER":      load_rds_B_matrix('output/gtex/PLIER_BP.rds'),
}

for name, B in embeddings.items():
    print(f"{name}: {B.shape}")

CLAMPbase: (578, 17382)
CLAMPfull: (289, 17382)
PLIER: (578, 17382)


## Core k-means function

In [44]:
def run_kmeans_best_ari(B, meta, label_col, min_samples, n_reps_per_k, multi_k=False):
    """
    Run GPU k-means and return the best mean ARI.

    multi_k=False  ->  k = [n_tissues]
    multi_k=True   ->  k = [n_tissues-10, n_tissues-5, n_tissues, n_tissues+5, n_tissues+10]
    """
    B = B.copy()
    B.columns = B.columns.astype(str).str.strip()

    common_ids = B.columns.intersection(meta.index)
    meta_f = meta.loc[common_ids]

    tissue_counts = meta_f[label_col].value_counts()
    valid_tissues = tissue_counts[tissue_counts >= min_samples].index
    mask = meta_f[label_col].isin(valid_tissues)
    meta_f = meta_f[mask]
    common_ids = meta_f.index

    n_tissues = len(valid_tissues)
    n_samples = len(common_ids)

    B_t = B.loc[:, common_ids].T.astype(np.float32)
    y_true = meta_f[label_col].astype(str).to_numpy()

    if multi_k:
        k_values = sorted({max(2, n_tissues + d) for d in [-10, -5, 0, 5, 10]})
    else:
        k_values = [n_tissues]

    best_overall_ari = -1
    best_k_overall   = None
    best_approach    = None

    for approach in ['scaled', 'unscaled']:
        X = StandardScaler().fit_transform(B_t) if approach == 'scaled' else B_t.values
        X_gpu = cp.asarray(X.astype('float32'))

        records = []
        for k in k_values:
            for rep in range(n_reps_per_k):
                km = cuKMeans(
                    n_clusters=k,
                    init='k-means++',
                    random_state=BASE_SEED + k * 1000 + rep,
                    max_iter=300,
                    tol=1e-4,
                    verbose=0,
                )
                labels = cp.asnumpy(km.fit_predict(X_gpu))
                records.append({'k': k, 'ari': adjusted_rand_score(y_true, labels)})

        del X_gpu
        cp.get_default_memory_pool().free_all_blocks()

        df_res   = pd.DataFrame(records)
        mean_ari = df_res.groupby('k')['ari'].mean()
        best_k   = mean_ari.idxmax()
        best_ari = mean_ari[best_k]

        if best_ari > best_overall_ari:
            best_overall_ari = best_ari
            best_k_overall   = best_k
            best_approach    = approach

    return {
        'best_ari':      float(best_overall_ari),
        'best_k':        int(best_k_overall),
        'best_approach': best_approach,
        'n_tissues':     int(n_tissues),
        'n_samples':     int(n_samples),
    }


def run_sweep(multi_k=False):
    rows = []
    param_combos = list(product(LABEL_COLS, MIN_SAMPLES_LIST, N_REPS_LIST))
    total = len(param_combos) * len(embeddings)
    done  = 0

    for label_col, min_samples, n_reps in param_combos:
        for model_name, B in embeddings.items():
            done += 1
            print(f"[{done}/{total}] {model_name} | {label_col} | min_samples={min_samples} | n_reps={n_reps}", end=" ", flush=True)

            res = run_kmeans_best_ari(
                B, gtex_meta,
                label_col=label_col,
                min_samples=min_samples,
                n_reps_per_k=n_reps,
                multi_k=multi_k,
            )

            print(f"| ARI={res['best_ari']:.4f}", flush=True)

            rows.append({
                'model':         model_name,
                'label_col':     label_col,
                'min_samples':   min_samples,
                'n_reps_per_k':  n_reps,
                'best_ari':      res['best_ari'],
                'best_k':        res['best_k'],
                'best_approach': res['best_approach'],
                'n_tissues':     res['n_tissues'],
                'n_samples':     res['n_samples'],
            })

    return pd.DataFrame(rows)

## Sweep 1 — k = n_tissues only

In [45]:
results_single_k = run_sweep(multi_k=False)
print("\nDone!")
results_single_k

[1/72] CLAMPbase | SMTS | min_samples=50 | n_reps=50 | ARI=0.6462
[2/72] CLAMPfull | SMTS | min_samples=50 | n_reps=50 | ARI=0.6393
[3/72] PLIER | SMTS | min_samples=50 | n_reps=50 | ARI=0.6651
[4/72] CLAMPbase | SMTS | min_samples=50 | n_reps=100 | ARI=0.6493
[5/72] CLAMPfull | SMTS | min_samples=50 | n_reps=100 | ARI=0.6345
[6/72] PLIER | SMTS | min_samples=50 | n_reps=100 | ARI=0.6649
[7/72] CLAMPbase | SMTS | min_samples=50 | n_reps=200 | ARI=0.6426
[8/72] CLAMPfull | SMTS | min_samples=50 | n_reps=200 | ARI=0.6335
[9/72] PLIER | SMTS | min_samples=50 | n_reps=200 | ARI=0.6720
[10/72] CLAMPbase | SMTS | min_samples=100 | n_reps=50 | ARI=0.6490
[11/72] CLAMPfull | SMTS | min_samples=100 | n_reps=50 | ARI=0.6441
[12/72] PLIER | SMTS | min_samples=100 | n_reps=50 | ARI=0.6753
[13/72] CLAMPbase | SMTS | min_samples=100 | n_reps=100 | ARI=0.6512
[14/72] CLAMPfull | SMTS | min_samples=100 | n_reps=100 | ARI=0.6421
[15/72] PLIER | SMTS | min_samples=100 | n_reps=100 | ARI=0.6779
[16/72] C

,model,label_col,min_samples,n_reps_per_k,best_ari,best_k,best_approach,n_tissues,n_samples
0,CLAMPbase,SMTS,50,50,0.646178,27,scaled,27,17333
1,CLAMPfull,SMTS,50,50,0.639256,27,scaled,27,17333
2,PLIER,SMTS,50,50,0.665077,27,scaled,27,17333
3,CLAMPbase,SMTS,50,100,0.649336,27,scaled,27,17333
4,CLAMPfull,SMTS,50,100,0.634507,27,scaled,27,17333
5,PLIER,SMTS,50,100,0.664852,27,scaled,27,17333
6,CLAMPbase,SMTS,50,200,0.642607,27,scaled,27,17333
7,CLAMPfull,SMTS,50,200,0.633534,27,scaled,27,17333
8,PLIER,SMTS,50,200,0.671964,27,scaled,27,17333
9,CLAMPbase,SMTS,100,50,0.648995,26,scaled,26,17244


## Sweep 2 — k = n_tissues ± 5, ± 10

In [46]:
results_multi_k = run_sweep(multi_k=True)
print("\nDone!")
results_multi_k

[1/72] CLAMPbase | SMTS | min_samples=50 | n_reps=50 | ARI=0.6690
[2/72] CLAMPfull | SMTS | min_samples=50 | n_reps=50 | ARI=0.6445
[3/72] PLIER | SMTS | min_samples=50 | n_reps=50 | ARI=0.6920
[4/72] CLAMPbase | SMTS | min_samples=50 | n_reps=100 | ARI=0.6638
[5/72] CLAMPfull | SMTS | min_samples=50 | n_reps=100 | ARI=0.6403
[6/72] PLIER | SMTS | min_samples=50 | n_reps=100 | ARI=0.6810
[7/72] CLAMPbase | SMTS | min_samples=50 | n_reps=200 | ARI=0.6659
[8/72] CLAMPfull | SMTS | min_samples=50 | n_reps=200 | ARI=0.6412
[9/72] PLIER | SMTS | min_samples=50 | n_reps=200 | ARI=0.6828
[10/72] CLAMPbase | SMTS | min_samples=100 | n_reps=50 | ARI=0.6675
[11/72] CLAMPfull | SMTS | min_samples=100 | n_reps=50 | ARI=0.6441
[12/72] PLIER | SMTS | min_samples=100 | n_reps=50 | ARI=0.6796
[13/72] CLAMPbase | SMTS | min_samples=100 | n_reps=100 | ARI=0.6669
[14/72] CLAMPfull | SMTS | min_samples=100 | n_reps=100 | ARI=0.6421
[15/72] PLIER | SMTS | min_samples=100 | n_reps=100 | ARI=0.6779
[16/72] C

,model,label_col,min_samples,n_reps_per_k,best_ari,best_k,best_approach,n_tissues,n_samples
0,CLAMPbase,SMTS,50,50,0.668979,22,scaled,27,17333
1,CLAMPfull,SMTS,50,50,0.644544,22,scaled,27,17333
2,PLIER,SMTS,50,50,0.692025,22,scaled,27,17333
3,CLAMPbase,SMTS,50,100,0.663808,22,scaled,27,17333
4,CLAMPfull,SMTS,50,100,0.640329,22,scaled,27,17333
5,PLIER,SMTS,50,100,0.681000,22,scaled,27,17333
6,CLAMPbase,SMTS,50,200,0.665943,22,scaled,27,17333
7,CLAMPfull,SMTS,50,200,0.641246,22,scaled,27,17333
8,PLIER,SMTS,50,200,0.682815,22,scaled,27,17333
9,CLAMPbase,SMTS,100,50,0.667486,21,scaled,26,17244


In [47]:
## Summary — best params per model

for label, df in [("single-k", results_single_k), ("multi-k", results_multi_k)]:
    print(f"\n--- {label} ---")
    for model in ['CLAMPbase', 'CLAMPfull', 'PLIER']:
        sub  = df[df['model'] == model]
        best = sub.loc[sub['best_ari'].idxmax()]
        print(f"  {model}: ARI={best['best_ari']:.4f} | label={best['label_col']} | "
              f"min_samples={best['min_samples']} | n_reps={best['n_reps_per_k']} | "
              f"k={best['best_k']} | approach={best['best_approach']}")


--- single-k ---
  CLAMPbase: ARI=0.6973 | label=SMTS | min_samples=200 | n_reps=50 | k=21 | approach=scaled
  CLAMPfull: ARI=0.7137 | label=SMTSD | min_samples=200 | n_reps=200 | k=37 | approach=scaled
  PLIER: ARI=0.7072 | label=SMTSD | min_samples=200 | n_reps=50 | k=37 | approach=scaled

--- multi-k ---
  CLAMPbase: ARI=0.6973 | label=SMTS | min_samples=200 | n_reps=50 | k=21 | approach=scaled
  CLAMPfull: ARI=0.7270 | label=SMTSD | min_samples=200 | n_reps=50 | k=32 | approach=scaled
  PLIER: ARI=0.7234 | label=SMTSD | min_samples=200 | n_reps=50 | k=32 | approach=scaled
